In [ ]:
%pip install gradio python-docx tiktoken

In [ ]:
import gradio as gr
import docx
import tiktoken
import os

def extract_text_with_formatting(doc_path):
    """
    Extrahiert Text aus einem Word-Dokument und wandelt grundlegende 
    Formatierungen (Überschriften, Fett, Kursiv) in Markdown um.
    """
    doc = docx.Document(doc_path)
    full_text = []

    # Text und Absatz-Formatierungen auslesen
    for para in doc.paragraphs:
        para_text = ""
        
        # Überschriften erkennen und als Markdown formatieren
        if para.style.name.startswith('Heading'):
            try:
                # "Heading 1" -> Level 1 -> "# "
                level = int(para.style.name.split(' ')[-1])
                para_text += ("#" * level) + " "
            except ValueError:
                para_text += "# "
        
        # Formatierungen innerhalb des Absatzes (Runs) erkennen
        for run in para.runs:
            text = run.text
            if not text.strip(): # Leerzeichen nicht formatieren
                para_text += text
                continue
                
            # Markdown-Formatierung hinzufügen
            if run.bold:
                text = f"**{text}**"
            if run.italic:
                text = f"_{text}_"
                
            para_text += text
            
        full_text.append(para_text)
        
    # Tabellen sehr simpel extrahieren
    if doc.tables:
        full_text.append("\n### Extrahierte Tabellen:\n")
        for table in doc.tables:
            for row in table.rows:
                row_data = [cell.text.replace('\n', ' ').strip() for cell in row.cells]
                full_text.append(" | ".join(row_data))
            full_text.append("-" * 20)

    return "\n".join(full_text)

def analyze_document(file):
    if file is None:
        return "Bitte lade ein Word-Dokument hoch.", ""

    try:
        # 1. Text und Formatierung extrahieren
        markdown_text = extract_text_with_formatting(file.name)
        
        # Reine Textversion (ohne Markdown) für den Vergleich
        doc_raw = docx.Document(file.name)
        raw_text = "\n".join([p.text for p in doc_raw.paragraphs])
        
        # 2. Tokenizer laden (cl100k_base wird von GPT-3.5/GPT-4 genutzt)
        encoding = tiktoken.get_encoding("cl100k_base")
        
        # 3. Zählungen durchführen
        markdown_tokens = len(encoding.encode(markdown_text))
        raw_tokens = len(encoding.encode(raw_text))
        word_count = len(raw_text.split())
        char_count = len(raw_text)
        
        # 4. Begründete Abschätzung formatieren
        analysis_report = f"""### 📊 Analyse-Ergebnis für: {os.path.basename(file.name)}

* **Wörter (ca.):** {word_count:,}
* **Zeichen (inkl. Leerzeichen):** {char_count:,}

#### 🧠 Token-Abschätzung (OpenAI cl100k_base Tokenizer)
* **Reiner Text:** {raw_tokens:,} Tokens
* **Text inkl. Formatierungen (Markdown):** {markdown_tokens:,} Tokens

**Begründung:** Ein Sprachmodell sieht das Dokument nicht als `.docx`, sondern als Text-String. Wenn du die Formatierungen (Fett, Kursiv, Überschriften, Tabellenstrukturen) in den Kontext übernehmen möchtest, um die semantische Struktur zu erhalten, müssen diese in Markdown übersetzt werden (z. B. `**Wichtiges Wort**`). 
Die Differenz von **{markdown_tokens - raw_tokens} Tokens** entsteht rein durch diese hinzugefügten Formatierungszeichen und Tabellen-Trennzeichen.

**Empfehlung:** Plane für dieses Dokument ein Context-Window-Budget von **ca. {markdown_tokens + int(markdown_tokens * 0.05)} Tokens** ein (inkl. 5% Puffer für eventuelle Metadaten oder System-Prompts, die den Dokumenten-Kontext begleiten).
"""
        return analysis_report, markdown_text

    except Exception as e:
        return f"❌ Fehler bei der Verarbeitung: {str(e)}", ""

# 5. Gradio Interface aufbauen
with gr.Blocks(title="Context Window Kalkulator", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 📄 Word-Dokument Token-Abschätzung")
    gr.Markdown("Lade ein `.docx` Dokument hoch, um zu sehen, wie viel Platz es im Context-Window eines LLMs verbrauchen wird.")
    
    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="Word-Dokument hochladen (.docx)", file_types=[".docx"])
            analyze_btn = gr.Button("Analysieren", variant="primary")
            
        with gr.Column(scale=2):
            result_output = gr.Markdown(label="Abschätzung")
            
    with gr.Accordion("Zeige extrahierten Markdown-Text (Wie das LLM ihn sieht)", open=False):
        text_preview = gr.Textbox(label="Extrahierter Code", lines=15, interactive=False)
        
    analyze_btn.click(
        fn=analyze_document,
        inputs=file_input,
        outputs=[result_output, text_preview]
    )


app.launch(inbrowser=True)